# Assignment 3 - Building a Simple Neural Network for Classification

## Data Preparation

### Import necessary libraries

In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import numpy as np
import os
import random

### Prepare to read and load data

In [63]:
def read_idx_images(filename):
    with open(filename, 'rb') as f:
        _, _, dims = struct.unpack('>HBB', f.read(4))
        shape = tuple(struct.unpack('>I', f.read(4))[0] for _ in range(dims))
        data = np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)
    return data

def read_idx_labels(filename):
    with open(filename, 'rb') as f:
        _, num_items = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

def load_data(images_path, labels_path, num_classes=10):
    images = read_idx_images(images_path).astype(np.float32) / 255.0
    labels = read_idx_labels(labels_path)
    images = images.reshape(images.shape[0], -1)
    labels_onehot = np.eye(num_classes)[labels].astype(np.float32)
    return torch.tensor(images), torch.tensor(labels_onehot)

### Load training and testing data

In [64]:
train_images = "train-images.idx3-ubyte"
train_labels = "train-labels.idx1-ubyte"
test_images  = "t10k-images.idx3-ubyte"
test_labels  = "t10k-labels.idx1-ubyte"

X_train, y_train = load_data(train_images, train_labels)
X_test, y_test = load_data(test_images, test_labels)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

## Model Architecture

### Create model Class

In [65]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_dim=784, hidden1=256, hidden2=128, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=1)
        return x

### Create CCE loss function

In [66]:
def categorical_cross_entropy(preds, targets, epsilon=1e-12):
 
    preds = torch.clamp(preds, epsilon, 1. - epsilon)
    ce = -torch.sum(targets * torch.log(preds), dim=1)
    return ce.mean()

### Add optimizer

In [67]:
device = torch.device("cpu")
model = FeedForwardNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## Model Training

### Create train and evaluate functions

In [68]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = categorical_cross_entropy(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = categorical_cross_entropy(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            targets = y_batch.argmax(dim=1)
            correct += (preds == targets).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

## Evaluation

### Run model and display evaluation scores

In [69]:
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    test_loss, test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch:02d}/{num_epochs} | Train Loss: {train_loss:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Epoch 01/10 | Train Loss: 0.2819 | Test Loss: 0.1276 | Test Acc: 0.9604
Epoch 02/10 | Train Loss: 0.1101 | Test Loss: 0.0922 | Test Acc: 0.9707
Epoch 03/10 | Train Loss: 0.0735 | Test Loss: 0.0956 | Test Acc: 0.9700
Epoch 04/10 | Train Loss: 0.0525 | Test Loss: 0.0696 | Test Acc: 0.9798
Epoch 05/10 | Train Loss: 0.0399 | Test Loss: 0.0725 | Test Acc: 0.9775
Epoch 06/10 | Train Loss: 0.0308 | Test Loss: 0.0659 | Test Acc: 0.9801
Epoch 07/10 | Train Loss: 0.0259 | Test Loss: 0.0796 | Test Acc: 0.9780
Epoch 08/10 | Train Loss: 0.0211 | Test Loss: 0.0883 | Test Acc: 0.9763
Epoch 09/10 | Train Loss: 0.0171 | Test Loss: 0.0908 | Test Acc: 0.9783
Epoch 10/10 | Train Loss: 0.0151 | Test Loss: 0.0858 | Test Acc: 0.9794


### Secondary test to show proof of prediction

In [70]:
model.eval()  # set to evaluation mode

# Pick a random index
idx = random.randint(0, len(X_test) - 1)
sample_image = X_test[idx].unsqueeze(0).to(device)  # shape (1, 784)
true_label = y_test[idx].argmax().item()

# Forward pass
with torch.no_grad():
    probs = model(sample_image)  # softmax outputs
    pred_label = probs.argmax(dim=1).item()

# Display results
print(f"True label: {true_label}")
print(f"Predicted label: {pred_label}")

True label: 7
Predicted label: 7
